Este notebook tem por objetivo avaliar as razões principais de churn relatadas dentro da variável CustomerFeedback, da base de dados history_tratada.

In [0]:
pip install seaborn

In [0]:
%sql
-- testando a importação das bases 
SELECT * from default.history
LIMIT 50

In [0]:
%sql

-- Todos os clientes tem serviços contratados?
SELECT * from default.history
WHERE PhoneService = 'No' AND InternetService = 'No' AND StreamingTV = 'No' AND StreamingMovies = 'No'
LIMIT 50

In [0]:
%sql
-- teste para segmentar churn
WITH base AS (
  SELECT
    *,
    try_cast(MonthlyCharges AS DECIMAL(10,2)) AS monthly_charges_num,

    CASE
      WHEN try_cast(tenure AS INT) <= 12 THEN '0–12 meses'
      WHEN try_cast(tenure AS INT) <= 36 THEN '13–36 meses'
      ELSE '37+ meses'
    END AS faixa_tenure,

    CASE
      WHEN try_cast(MonthlyCharges AS DECIMAL(10,2)) < 35 THEN 'Baixo valor mensal'
      WHEN try_cast(MonthlyCharges AS DECIMAL(10,2)) < 75 THEN 'Médio valor mensal'
      ELSE 'Alto valor mensal'
    END AS faixa_valor_mensal,

    CASE
      WHEN PaymentMethod IN (
        'Bank transfer (automatic)',
        'Credit card (automatic)'
      ) THEN 'Automático'
      ELSE 'Manual'
    END AS tipo_pagamento
  FROM default.history
)

SELECT
  faixa_tenure,
  faixa_valor_mensal,
  Contract,
  tipo_pagamento,
  COUNT(*) AS clientes,
  ROUND(
    100 * AVG(CASE WHEN Churn = 'Yes' THEN 1.0 ELSE 0.0 END),
    2
  ) AS churn_pct
FROM base
GROUP BY 1, 2, 3, 4
ORDER BY churn_pct DESC;

In [0]:
%sql
-- teste para segmentar churn
WITH segmentos AS (
  SELECT
    faixa_tenure,
    faixa_valor_mensal,
    Contract,
    tipo_pagamento,
    clientes,
    churn_pct
  FROM _sqldf
)

SELECT
  concat(
    faixa_tenure, ' | ',
    faixa_valor_mensal, ' | ',
    Contract, ' | ',
    tipo_pagamento
  ) AS segmento,
  clientes,
  churn_pct,
  ROUND(clientes * churn_pct / 100, 0) AS churns_estimados
FROM segmentos
WHERE clientes >= 30
ORDER BY churns_estimados DESC;

Pensando em Features que poderiam auxiliar na análise e na modelagem também:

- agregação de pagamento em meios automáticos 
- qual porcentagem do income o custo com os serviços da Terra Signal geralmente custa?
- dentro do mesmo serviço, existe grande variação dessa porcentagem?

In [0]:
%sql

-- Criando as features especuladas em uma tabela de features auxiliares, contendo a key customerID.
CREATE OR REPLACE TABLE default.aux_features AS

WITH base AS (
  SELECT
    customerID,

-- pagamento automático
    CASE
      WHEN PaymentMethod IN (
        'Bank transfer (automatic)',
        'Credit card (automatic)'
      ) THEN 1
      ELSE 0
    END AS pagamento_automatico,

-- porcentagem de renda do cliente destinada a Terra Signal - o quanto pesa no bolso do cliente mensalmente 
    ROUND(
      100.0 * try_cast(MonthlyCharges AS DECIMAL(12,2))
      / NULLIF(try_cast(MonthlyIncome AS DECIMAL(12,2)), 0),
      2
    ) AS pct_renda_terra_signal,

    concat_ws(
      ' | ',
      PhoneService,
      MultipleLines,
      InternetService,
      OnlineSecurity,
      OnlineBackup,
      DeviceProtection,
      TechSupport,
      StreamingTV,
      StreamingMovies
    ) AS pacote

  FROM default.history_tratado
),

medianas_pacote AS (
  SELECT
    pacote,
    COUNT(*) AS clientes_no_pacote,
    percentile_approx(pct_renda_terra_signal, 0.5) AS pct_renda_mediana_pacote
  FROM base
  GROUP BY pacote
)

SELECT
  b.customerID,
  b.pagamento_automatico,
  b.pct_renda_terra_signal,

-- O quanto a cobrança para este cliente é deslocada da mediana da cobrança para o mesmo pacote oferido. Calculado para grupos com ao menos 30 clientes inclusos.
  CASE
    WHEN m.clientes_no_pacote >= 30
    THEN ROUND(
      b.pct_renda_terra_signal - m.pct_renda_mediana_pacote,
      2
    )
  END AS diferenca_pct_renda_vs_mediana_pacote

FROM base b
LEFT JOIN medianas_pacote m
  ON b.pacote = m.pacote;

In [0]:
%sql
-- validaçào da tabela criada com as features auxiliares.
SELECT *
FROM default.aux_features
LIMIT 50;

In [0]:
%sql
SELECT
  h.*,
  f.pagamento_automatico,
  f.pct_renda_terra_signal,
  f.diferenca_pct_renda_vs_mediana_pacote
FROM default.history_tratado h
LEFT JOIN default.aux_features f
  ON h.customerID = f.customerID;

In [0]:
%sql
-- criando grupos de renda por porcentagem para estimar correlação com churn
WITH base AS (
  SELECT
    h.Churn,

    CASE
      WHEN f.pct_renda_terra_signal < 1 THEN '< 1%'
      WHEN f.pct_renda_terra_signal < 3 THEN '1%–3%'
      WHEN f.pct_renda_terra_signal < 5 THEN '3%–5%'
      WHEN f.pct_renda_terra_signal < 10 THEN '5%–10%'
      ELSE '> 10%'
    END AS faixa_pct_renda,

    CASE
      WHEN f.pct_renda_terra_signal < 1 THEN 1
      WHEN f.pct_renda_terra_signal < 3 THEN 2
      WHEN f.pct_renda_terra_signal < 5 THEN 3
      WHEN f.pct_renda_terra_signal < 10 THEN 4
      ELSE 5
    END AS ordem_faixa,

    CASE
      WHEN f.pagamento_automatico = 1 THEN 'Automático'
      ELSE 'Manual'
    END AS tipo_pagamento

  FROM default.history_tratado h
  INNER JOIN default.aux_features f
    ON h.customerID = f.customerID
  WHERE f.pct_renda_terra_signal IS NOT NULL
)

SELECT
  faixa_pct_renda,
  ordem_faixa,
  tipo_pagamento,
  COUNT(*) AS clientes,
  ROUND(
    100.0 * AVG(CASE WHEN Churn = 'Yes' THEN 1.0 ELSE 0.0 END),
    2
  ) AS taxa_churn_pct
FROM base
GROUP BY
  faixa_pct_renda,
  ordem_faixa,
  tipo_pagamento
ORDER BY
  ordem_faixa,
  tipo_pagamento;

In [0]:
%python
## Explorando a correlação das features com o churn

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Une a base histórica com as features auxiliares
df = (
    spark.table("default.history_tratado")
    .join(
        spark.table("default.aux_features"),
        on="customerID",
        how="inner"
    )
    .toPandas()
)

# Target: Yes = 1, No = 0
df["churn_binario"] = (
    df["Churn"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({"yes": 1, "no": 0})
)

def normaliza(coluna):
    return df[coluna].astype(str).str.strip().str.lower()

# Features numéricas antigas
features = pd.DataFrame({
    "SeniorCitizen": pd.to_numeric(df["SeniorCitizen"], errors="coerce"),
    "tenure": pd.to_numeric(df["tenure"], errors="coerce"),
    "MonthlyCharges": pd.to_numeric(df["MonthlyCharges"], errors="coerce"),
    "TotalCharges": pd.to_numeric(df["TotalCharges"], errors="coerce"),
    "MonthlyIncome": pd.to_numeric(df["MonthlyIncome"], errors="coerce"),
})

# Features binárias antigas
colunas_sim_nao = [
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "PaperlessBilling",
]

for coluna in colunas_sim_nao:
    features[f"{coluna}_yes"] = normaliza(coluna).eq("yes").astype(int)

features["gender_female"] = normaliza("gender").eq("female").astype(int)

# Dummies das categorias antigas
categoricas = ["InternetService", "Contract", "PaymentMethod"]

dummies = pd.get_dummies(
    df[categoricas].astype(str),
    prefix=categoricas,
    dtype=int
)

features = pd.concat([features, dummies], axis=1)

# Features novas
features["pagamento_automatico"] = pd.to_numeric(
    df["pagamento_automatico"], errors="coerce"
)

features["pct_renda_terra_signal"] = pd.to_numeric(
    df["pct_renda_terra_signal"], errors="coerce"
)

features["diferenca_pct_renda_vs_mediana_pacote"] = pd.to_numeric(
    df["diferenca_pct_renda_vs_mediana_pacote"], errors="coerce"
)

# Inclui o alvo e remove features sem variação
features["Churn"] = df["churn_binario"]
features = features.loc[:, features.nunique(dropna=True) > 1]

# Correlação de cada feature com Churn
correlacoes = (
    features.corr(method="pearson")["Churn"]
    .drop("Churn")
    .sort_values(key=lambda x: x.abs(), ascending=False)
    .to_frame(name="Correlação com churn")
)

# Heatmap
plt.figure(figsize=(10, max(8, len(correlacoes) * 0.35)))

sns.heatmap(
    correlacoes,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn_r",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    cbar_kws={"label": "Correlação de Pearson"}
)

plt.title("Correlação das features com Churn")
plt.xlabel("")
plt.ylabel("Features")
plt.tight_layout()
plt.show()


### Análise do Feedback dos clientes

In [0]:
%sql
-- criar tabela para analisar os feedbacks

CREATE OR REPLACE TABLE default.feedback_base AS

SELECT
  h.customerID,
  h.Churn,

  -- Texto original, preservado para inspeção posterior
  h.CustomerFeedback AS feedback_original,

  -- Texto limpo para análise
  lower(
    trim(
      regexp_replace(
        h.CustomerFeedback,
        '[\\r\\n]+',
        ' '
      )
    )
  ) AS feedback_limpo,

  -- Métrica básica do texto
  length(
    trim(
      regexp_replace(h.CustomerFeedback, '[\\r\\n]+', ' ')
    )
  ) AS tamanho_feedback,

  -- Variáveis para cruzamentos posteriores
  try_cast(h.MonthlyCharges AS DECIMAL(12,2)) AS monthly_charges,
  try_cast(h.MonthlyIncome AS DECIMAL(12,2)) AS monthly_income,
  h.InternetService,
  h.Contract,
  h.PaymentMethod,

  -- Features já criadas
  f.pagamento_automatico,
  f.pct_renda_terra_signal,
  f.diferenca_pct_renda_vs_mediana_pacote

FROM default.history_tratado h
LEFT JOIN default.aux_features f
  ON h.customerID = f.customerID
WHERE h.CustomerFeedback IS NOT NULL
  AND trim(h.CustomerFeedback) <> '';

In [0]:
%sql

-- validando a base

SELECT
  Churn,
  COUNT(*) AS clientes_com_feedback,
  ROUND(AVG(tamanho_feedback), 0) AS tamanho_medio_caracteres,
  MIN(tamanho_feedback) AS menor_feedback,
  MAX(tamanho_feedback) AS maior_feedback
FROM default.feedback_base
GROUP BY Churn;

In [0]:
%sql
-- testando exemplos de churn 
SELECT
  customerID,
  InternetService,
  Contract,
  monthly_charges,
  pct_renda_terra_signal,
  feedback_original
FROM default.feedback_base
WHERE Churn = 'Yes'
LIMIT 20;

In [0]:
%sql
-- os feedbacks são bastante estruturados para análise dos motivos do churn. Assim seguiremos uma abordagem simples criando categorias em SQL com regex antes de qualquer aplicação de IA generativa

-- investigando os motivos

CREATE OR REPLACE TABLE default.feedback_churn_motivos AS

WITH sinais AS (
  SELECT
    *,

    CASE
      WHEN feedback_limpo RLIKE
        'better deal|better value|more affordable|cost savings|too expensive|high monthly charges|charges.*too high|competitive pric|cost considerations|cheaper|pricing.*(led|reason|ultimately)'
      THEN 1 ELSE 0
    END AS motivo_preco,

    CASE
      WHEN feedback_limpo RLIKE
        'slow speed|unreliable|connectivity issue|connection issue|connection problem|service did not meet|speed.*did not meet|outage'
      THEN 1 ELSE 0
    END AS motivo_tecnico,

    CASE
      WHEN feedback_limpo RLIKE
        '(electronic check|mailed check|bank transfer|credit card).*(inconvenient|issue|problem|hassle)'
        OR feedback_limpo RLIKE
        'payment method.*(inconvenient|issue|problem|hassle)'
        OR feedback_limpo RLIKE
        'payment.*did not go through'
      THEN 1 ELSE 0
    END AS motivo_pagamento,

    CASE
      WHEN feedback_limpo RLIKE
        'lack of internet service.*(inconvenience|inconvenient|not satisfied|disappointed)'
        OR feedback_limpo RLIKE
        'no internet service.*(inconvenience|inconvenient|not satisfied|disappointed)'
      THEN 1 ELSE 0
    END AS motivo_produto,

    CASE
      WHEN feedback_limpo RLIKE
        'personal reasons|financial issues|moving|move to|relocat'
      THEN 1 ELSE 0
    END AS motivo_pessoal,

    CASE
      WHEN feedback_limpo RLIKE
        'customer service'
      THEN 1 ELSE 0
    END AS motivo_atendimento

  FROM default.feedback_base
  WHERE Churn = 'Yes'
)

SELECT
  *,

  CASE
    WHEN motivo_preco = 1 THEN 'Preço / oferta concorrente'
    WHEN motivo_tecnico = 1 THEN 'Qualidade técnica do serviço'
    WHEN motivo_produto = 1 THEN 'Limitação de produto'
    WHEN motivo_pagamento = 1 THEN 'Experiência de pagamento'
    WHEN motivo_pessoal = 1 THEN 'Motivo pessoal/financeiro'
    WHEN motivo_atendimento = 1 THEN 'Atendimento ao cliente'
    ELSE 'Não identificado'
  END AS motivo_principal

FROM sinais;

In [0]:
%sql
-- contagem de motivos
SELECT
  motivo_principal,
  COUNT(*) AS clientes_churn,
  ROUND(
    100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
    2
  ) AS percentual_dos_churns
FROM default.feedback_churn_motivos
GROUP BY motivo_principal
ORDER BY clientes_churn DESC;

In [0]:
%sql
-- avaliando a classificação levando em conta os clientes que relataram mais de um motivo para o churn
CREATE OR REPLACE TABLE default.feedback_churn_motivos_long AS

SELECT customerID, 'Preço / oferta concorrente' AS motivo
FROM default.feedback_churn_motivos
WHERE motivo_preco = 1

UNION ALL

SELECT customerID, 'Qualidade técnica do serviço' AS motivo
FROM default.feedback_churn_motivos
WHERE motivo_tecnico = 1

UNION ALL

SELECT customerID, 'Experiência de pagamento' AS motivo
FROM default.feedback_churn_motivos
WHERE motivo_pagamento = 1

UNION ALL

SELECT customerID, 'Limitação de produto' AS motivo
FROM default.feedback_churn_motivos
WHERE motivo_produto = 1

UNION ALL

SELECT customerID, 'Motivo pessoal/financeiro' AS motivo
FROM default.feedback_churn_motivos
WHERE motivo_pessoal = 1

UNION ALL

SELECT customerID, 'Atendimento ao cliente' AS motivo
FROM default.feedback_churn_motivos
WHERE motivo_atendimento = 1;

In [0]:
%sql
-- ocorrência dos motivos para classificação 
SELECT
  motivo,
  COUNT(DISTINCT customerID) AS clientes_churn,
  ROUND(
    100.0 * COUNT(DISTINCT customerID)
    / (SELECT COUNT(*) FROM default.feedback_churn_motivos),
    2
  ) AS pct_dos_clientes_com_churn
FROM default.feedback_churn_motivos_long
GROUP BY motivo
ORDER BY clientes_churn DESC;

### Qualidade da avaliação

In [0]:
%sql
-- criando amostras pra avaliação de classificação
-- foram separados 25 exemplos aleatórios para cada motivo, e eles serão avaliados para seu fit com a classificação

CREATE OR REPLACE TABLE default.auditoria_motivos_feedback AS

WITH amostra AS (
  SELECT
    m.motivo,
    f.customerID,
    f.InternetService,
    f.Contract,
    f.monthly_charges,
    f.pct_renda_terra_signal,
    f.feedback_original,

    ROW_NUMBER() OVER (
      PARTITION BY m.motivo
      ORDER BY rand(42)
    ) AS numero_na_amostra

  FROM default.feedback_churn_motivos_long m
  INNER JOIN default.feedback_churn_motivos f
    ON m.customerID = f.customerID
)

SELECT
  motivo AS motivo_previsto,
  customerID,
  InternetService,
  Contract,
  monthly_charges,
  pct_renda_terra_signal,
  feedback_original,

  CAST(NULL AS STRING) AS motivo_validado,
  CAST(NULL AS STRING) AS classificacao_correta,
  CAST(NULL AS STRING) AS observacao

FROM amostra
WHERE numero_na_amostra <= 25;

In [0]:
%sql
-- PREÇO/CONCORRENTE
SELECT *
FROM default.auditoria_motivos_feedback
WHERE motivo_previsto = 'Preço / oferta concorrente';

In [0]:
%sql
--- adicionando avaliação para os casos de preço/concorrência
UPDATE default.auditoria_motivos_feedback
SET
  motivo_validado = 'Preço / oferta concorrente',
  classificacao_correta = 'Sim',
  observacao = 'Preço alto, economia, melhor oferta ou melhor valor do concorrente mencionado explicitamente.'
WHERE motivo_previsto = 'Preço / oferta concorrente'
  AND customerID IN (
    '3351-NQLDI',
    '0354-WYROK',
    '2645-QTLMB',
    '0870-VEMYL',
    '3858-VOBET',
    '0207-MDKNV',
    '6015-VVHHE',
    '2141-RRYGO',
    '8780-IXSTS',
    '1465-VINDH',
    '8058-JMEQO',
    '1862-SKORY',
    '9408-SSNVZ',
    '3318-OSATS',
    '3398-FSHON',
    '5020-ZSTTY',
    '9369-XFEHK',
    '1193-RTSLK',
    '5073-WXOYN',
    '3874-EQOEP',
    '1075-BGWOH',
    '8417-FMLZI',
    '6023-YEBUP',
    '0564-JJHGS',
    '3884-HCSWG'
  );

In [0]:
%sql
-- QUALIDADE SERVIÇO
SELECT *
FROM default.auditoria_motivos_feedback
WHERE motivo_previsto = 'Qualidade técnica do serviço';

In [0]:
%sql
UPDATE default.auditoria_motivos_feedback
SET
  motivo_validado = 'Qualidade técnica do serviço',
  classificacao_correta = 'Sim',
  observacao = 'Lentidão, instabilidade, interrupções ou baixa confiabilidade mencionadas explicitamente como motivo de churn.'
WHERE motivo_previsto = 'Qualidade técnica do serviço'
  AND customerID IN (
    '0723-DRCLG',
    '1112-CUNAO',
    '3727-RJMEO',
    '9546-CQJSU',
    '0822-GAVAP',
    '4190-MFLUW',
    '3891-PUQOD',
    '8348-HFYIV',
    '6711-VTNRE',
    '8563-IIOXK',
    '4369-HTUIF',
    '4503-BDXBD',
    '5378-IKEEG',
    '6680-WKXRZ',
    '4629-NRXKX',
    '4566-QVRRW',
    '5561-NWEVX',
    '5494-HECPR',
    '7802-EFKNY',
    '8473-VUVJN',
    '6633-MPWBS',
    '1450-SKCVI',
    '7881-EVUAD',
    '5797-APWZC',
    '2454-RPBRZ'
  );

In [0]:
%sql
-- MÉTODO DE PAGAMENTO
SELECT *
FROM default.auditoria_motivos_feedback
WHERE motivo_previsto = 'Experiência de pagamento';

Encontramos uma falha na classificação do motivo de falha no pagamento.
Assim, vamos anotar os exemplos e corrigir a atribuição de motivo pra essa categoria.

In [0]:
%sql
UPDATE default.auditoria_motivos_feedback
SET
  motivo_validado = 'Experiência de pagamento',
  classificacao_correta = 'Sim',
  observacao = 'Inconveniência, atraso, falha ou falta de opções de pagamento mencionada explicitamente.'
WHERE motivo_previsto = 'Experiência de pagamento'
  AND customerID IN (
    '5515-IDEJJ',
    '4759-TRPLW',
    '3049-NDXFL',
    '1240-KNSEZ',
    '6502-MJQAE',
    '8802-UNOJF',
    '2037-SGXHH',
    '9689-PTNPG',
    '4946-EDSEW',
    '4951-UKAAQ',
    '8943-URTMR',
    '5688-KZTSN',
    '4877-EVATK',
    '4625-EWPTF'
  );

In [0]:
%sql
MERGE INTO default.auditoria_motivos_feedback destino

USING (
  SELECT
    '2272-QAGFO' AS customerID,
    'Preço / oferta concorrente' AS motivo_real

  UNION ALL SELECT '0689-NKYLF', 'Preço / oferta concorrente'
  UNION ALL SELECT '3746-EUBYR', 'Motivo pessoal/financeiro'
  UNION ALL SELECT '2479-BRAMR', 'Não identificado'
  UNION ALL SELECT '8679-JOEVF', 'Preço / oferta concorrente'
  UNION ALL SELECT '4315-MURBD', 'Preço / oferta concorrente'
  UNION ALL SELECT '0023-XUOPT', 'Preço / oferta concorrente'
  UNION ALL SELECT '8837-VVWLQ', 'Qualidade técnica do serviço'
  UNION ALL SELECT '4547-KQRTM', 'Preço / oferta concorrente'
  UNION ALL SELECT '8058-INTPH', 'Qualidade técnica do serviço'
  UNION ALL SELECT '6821-BUXUX', 'Preço / oferta concorrente'
) origem

ON destino.customerID = origem.customerID
AND destino.motivo_previsto = 'Experiência de pagamento'

WHEN MATCHED THEN UPDATE SET
  destino.motivo_validado = origem.motivo_real,
  destino.classificacao_correta = 'Não',
  destino.observacao =
    'Método de pagamento apenas citado como conveniente ou sem relação causal com o churn.';

In [0]:
%sql
SELECT
  classificacao_correta,
  COUNT(*) AS quantidade
FROM default.auditoria_motivos_feedback
WHERE motivo_previsto = 'Experiência de pagamento'
GROUP BY classificacao_correta;

### atualizando motivo de pagamento


In [0]:
%sql
UPDATE default.feedback_churn_motivos
SET motivo_pagamento =
  CASE

    -- Exceções: frases que dizem explicitamente que NÃO houve problema
    WHEN feedback_limpo RLIKE
      '(never had|did not encounter|no).{0,30}(issues?|problems?) with billing'
    THEN 0

    -- Meio de pagamento inconveniente, limitado ou ultrapassado
    WHEN feedback_limpo RLIKE
      '(payment method|payment options?|billing options?|electronic check|mailed check|bank transfer|credit card).{0,60}(inconvenient|outdated|limited|lack of flexibility|not flexible|more flexible|more convenient|hassle(?!-free))'

    OR feedback_limpo RLIKE
      '(inconvenient|outdated|limited|lack of flexibility|not flexible|hassle(?!-free)).{0,60}(payment method|payment options?|electronic check|mailed check|bank transfer|credit card)'

    -- Falhas explícitas de processamento
    OR feedback_limpo RLIKE
      '(payment|electronic check|mailed check|bank transfer|credit card).{0,60}(did not go through|didn.t go through|not processed|not always processed|took longer than expected|not processed in a timely manner)'

    -- Problemas diretamente associados ao meio de pagamento
    OR feedback_limpo RLIKE
      '(issues?|problems?|delays?).{0,30}(with|using|in).{0,30}(payment method|electronic check|mailed check|bank transfer|credit card)'

    -- Problemas de billing sem a expressão negativa tratada acima
    OR feedback_limpo RLIKE
      '(billing issues|billing problems|billing delays|issues with billing|problems with billing|delays with billing)'

    -- Obrigação de usar cheque enviado pelo correio
    OR feedback_limpo RLIKE
      '(had to|having to|limited to).{0,40}(mailing a check|mailed check)'

    -- Preferência explícita por outro meio
    OR feedback_limpo RLIKE
      'prefer.{0,60}(credit card|online payment|automatic payment).{0,80}(rather than|instead of|over).{0,50}(electronic check|mailed check)'

    THEN 1
    ELSE 0
  END;

In [0]:
%sql
SELECT
  a.classificacao_correta,
  m.motivo_pagamento,
  COUNT(*) AS quantidade
FROM default.auditoria_motivos_feedback a
INNER JOIN default.feedback_churn_motivos m
  ON a.customerID = m.customerID
WHERE a.motivo_previsto = 'Experiência de pagamento'
GROUP BY
  a.classificacao_correta,
  m.motivo_pagamento
ORDER BY
  a.classificacao_correta,
  m.motivo_pagamento;

In [0]:
%sql
UPDATE default.feedback_churn_motivos
SET motivo_principal =
  CASE
    WHEN motivo_preco = 1 THEN 'Preço / oferta concorrente'
    WHEN motivo_tecnico = 1 THEN 'Qualidade técnica do serviço'
    WHEN motivo_produto = 1 THEN 'Limitação de produto'
    WHEN motivo_pagamento = 1 THEN 'Experiência de pagamento'
    WHEN motivo_pessoal = 1 THEN 'Motivo pessoal/financeiro'
    WHEN motivo_atendimento = 1 THEN 'Atendimento ao cliente'
    ELSE 'Não identificado'
  END;

In [0]:
%sql
CREATE OR REPLACE TABLE default.feedback_churn_motivos_long AS

SELECT customerID, 'Preço / oferta concorrente' AS motivo
FROM default.feedback_churn_motivos
WHERE motivo_preco = 1

UNION ALL

SELECT customerID, 'Qualidade técnica do serviço' AS motivo
FROM default.feedback_churn_motivos
WHERE motivo_tecnico = 1

UNION ALL

SELECT customerID, 'Experiência de pagamento' AS motivo
FROM default.feedback_churn_motivos
WHERE motivo_pagamento = 1

UNION ALL

SELECT customerID, 'Limitação de produto' AS motivo
FROM default.feedback_churn_motivos
WHERE motivo_produto = 1

UNION ALL

SELECT customerID, 'Motivo pessoal/financeiro' AS motivo
FROM default.feedback_churn_motivos
WHERE motivo_pessoal = 1

UNION ALL

SELECT customerID, 'Atendimento ao cliente' AS motivo
FROM default.feedback_churn_motivos
WHERE motivo_atendimento = 1;

In [0]:
%sql
SELECT
  motivo,
  COUNT(DISTINCT customerID) AS clientes_churn,
  ROUND(
    100.0 * COUNT(DISTINCT customerID)
      / (SELECT COUNT(*) FROM default.feedback_churn_motivos),
    2
  ) AS pct_dos_churns
FROM default.feedback_churn_motivos_long
GROUP BY motivo
ORDER BY clientes_churn DESC;

In [0]:
%sql
SELECT
  a.customerID,
  a.feedback_original,
  a.observacao,
  m.motivo_pagamento
FROM default.auditoria_motivos_feedback a
INNER JOIN default.feedback_churn_motivos m
  ON a.customerID = m.customerID
WHERE a.motivo_previsto = 'Experiência de pagamento'
  AND a.classificacao_correta = 'Não'
  AND m.motivo_pagamento = 1;

In [0]:
%sql
SELECT
  CASE
    WHEN motivo_pagamento = 1
      AND motivo_preco = 0
      AND motivo_tecnico = 0
      AND motivo_produto = 0
      AND motivo_pessoal = 0
      AND motivo_atendimento = 0
    THEN 'Pagamento como motivo isolado'

    WHEN motivo_pagamento = 1
    THEN 'Pagamento combinado com outro motivo'
  END AS tipo_motivo_pagamento,

  COUNT(*) AS clientes_churn,

  ROUND(
    100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
    2
  ) AS percentual_dentro_dos_motivos_de_pagamento

FROM default.feedback_churn_motivos
WHERE motivo_pagamento = 1
GROUP BY 1
ORDER BY clientes_churn DESC;

### avaliando limitação de serviços para clientes sem internet

In [0]:
%sql
SELECT
  customerID,
  InternetService,
  Contract,
  monthly_charges,
  feedback_original
FROM default.feedback_churn_motivos
WHERE InternetService = 'No'
  AND motivo_produto = 0
  AND feedback_limpo RLIKE
    'not included|not available|not provided|without internet|no access to internet|lack of internet'
ORDER BY customerID;

In [0]:
%sql
SELECT
  CASE
    WHEN motivo_produto = 1 THEN 'Já classificado como limitação de produto'

    WHEN feedback_limpo RLIKE
      'not included|not available|not provided|without internet|no access to internet|lack of internet'
    THEN 'Candidato a limitação de produto'

    ELSE 'Sem sinal de limitação de produto'
  END AS classificacao,

  COUNT(*) AS clientes_churn

FROM default.feedback_churn_motivos
WHERE InternetService = 'No'
GROUP BY 1
ORDER BY clientes_churn DESC;

In [0]:
%sql
SELECT
  customerID,
  Contract,
  monthly_charges,
  pct_renda_terra_signal,
  feedback_original
FROM default.feedback_churn_motivos
WHERE InternetService = 'No'
  AND COALESCE(motivo_produto, 0) = 0
  AND feedback_limpo RLIKE
    'not included|not available|not provided|without internet|no access to internet|lack of internet'
ORDER BY customerID;

In [0]:
%sql

-- ATUALIZANDO REGRA Limitação de Produto/Cobertura

UPDATE default.feedback_churn_motivos
SET motivo_produto =
  CASE
    WHEN motivo_produto = 1 THEN 1

    WHEN feedback_limpo RLIKE
      'lack of internet service|lack of internet access|without internet service|without internet access|no access to internet|did not have internet service|internet service was not included|internet service not included|internet service was not available|internet service not available|internet service was not provided|internet service not provided|did not offer internet service|not offer internet service|lack of internet service options|not available in my area|internet service options available'

    THEN 1
    ELSE 0
  END;

In [0]:
%sql
UPDATE default.feedback_churn_motivos
SET motivo_principal =
  CASE
    WHEN motivo_preco = 1 THEN 'Preço / oferta concorrente'
    WHEN motivo_tecnico = 1 THEN 'Qualidade técnica do serviço'
    WHEN motivo_produto = 1 THEN 'Limitação de produto ou cobertura'
    WHEN motivo_pagamento = 1 THEN 'Experiência de pagamento'
    WHEN motivo_pessoal = 1 THEN 'Motivo pessoal/financeiro'
    WHEN motivo_atendimento = 1 THEN 'Atendimento ao cliente'
    ELSE 'Não identificado'
  END;

In [0]:
%sql
-- 3. Recria a tabela multirrótulo completa
-- Um mesmo customerID pode aparecer em mais de uma categoria.
CREATE OR REPLACE TABLE default.feedback_churn_motivos_long AS

SELECT
  customerID,
  'Preço / oferta concorrente' AS motivo
FROM default.feedback_churn_motivos
WHERE motivo_preco = 1

UNION ALL

SELECT
  customerID,
  'Qualidade técnica do serviço' AS motivo
FROM default.feedback_churn_motivos
WHERE motivo_tecnico = 1

UNION ALL

SELECT
  customerID,
  'Experiência de pagamento' AS motivo
FROM default.feedback_churn_motivos
WHERE motivo_pagamento = 1

UNION ALL

SELECT
  customerID,
  'Limitação de produto ou cobertura' AS motivo
FROM default.feedback_churn_motivos
WHERE motivo_produto = 1

UNION ALL

SELECT
  customerID,
  'Motivo pessoal/financeiro' AS motivo
FROM default.feedback_churn_motivos
WHERE motivo_pessoal = 1

UNION ALL

SELECT
  customerID,
  'Atendimento ao cliente' AS motivo
FROM default.feedback_churn_motivos
WHERE motivo_atendimento = 1;

In [0]:
%sql
SELECT
  motivo,
  COUNT(DISTINCT customerID) AS clientes_churn,
  ROUND(
    100.0 * COUNT(DISTINCT customerID)
      / (SELECT COUNT(*) FROM default.feedback_churn_motivos),
    2
  ) AS pct_dos_churns
FROM default.feedback_churn_motivos_long
GROUP BY motivo
ORDER BY clientes_churn DESC;

In [0]:
%sql
SELECT
  InternetService,
  COUNT(*) AS clientes_churn_produto_cobertura,
  ROUND(
    100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
    2
  ) AS percentual_na_categoria
FROM default.feedback_churn_motivos
WHERE motivo_produto = 1
GROUP BY InternetService
ORDER BY clientes_churn_produto_cobertura DESC;

## Especificação por motivos


In [0]:
%sql
-- Dentro de preço/concorrência, vamos avaliar melhor investigar se o motivo é preço alto ou melhor preço na concorrência

CREATE OR REPLACE TABLE default.feedback_churn_preco_detalhe AS

WITH base AS (
  SELECT
    f.customerID,
    f.feedback_limpo,
    f.monthly_charges,
    f.monthly_income,
    f.pct_renda_terra_signal,
    f.InternetService,
    f.Contract,
    f.PaymentMethod,
    h.tenure,

    CASE
      WHEN f.feedback_limpo RLIKE
        'high monthly charges|charges.*too high|monthly charges.*too high|too expensive|bit on the higher side|pricing.*too high'
      THEN 1 ELSE 0
    END AS preco_percebido_alto,

    CASE
      WHEN f.feedback_limpo RLIKE
        'better deal|better value|more affordable|cost savings|competitive pric|cheaper'
      THEN 1 ELSE 0
    END AS oferta_concorrente

  FROM default.feedback_base f
  INNER JOIN default.feedback_churn_motivos m
    ON f.customerID = m.customerID
  INNER JOIN default.history_tratado h
    ON f.customerID = h.customerID
  WHERE m.motivo_preco = 1
)

SELECT
  *,

  CASE
    WHEN preco_percebido_alto = 1
      AND oferta_concorrente = 1
      THEN 'Preço alto + oferta concorrente'

    WHEN preco_percebido_alto = 1
      THEN 'Preço percebido alto'

    WHEN oferta_concorrente = 1
      THEN 'Melhor oferta concorrente'

    ELSE 'Motivo de preço não especificado'
  END AS tipo_motivo_preco

FROM base;

In [0]:
%sql
SELECT
  tipo_motivo_preco,

  COUNT(*) AS clientes_churn,

  ROUND(AVG(monthly_charges), 2) AS mensalidade_media,
  ROUND(percentile_approx(monthly_charges, 0.5), 2) AS mediana_mensalidade,

  ROUND(AVG(monthly_income), 2) AS renda_media,

  ROUND(AVG(pct_renda_terra_signal), 2) AS pct_renda_medio,
  ROUND(
    percentile_approx(pct_renda_terra_signal, 0.5),
    2
  ) AS mediana_pct_renda,

  ROUND(AVG(try_cast(tenure AS DOUBLE)), 1) AS tenure_medio

FROM default.feedback_churn_preco_detalhe
GROUP BY tipo_motivo_preco
ORDER BY clientes_churn DESC;

### Preço

In [0]:
%sql

-- investigando os motivo de preço não especificado
SELECT
  customerID,
  monthly_charges,
  monthly_income,
  pct_renda_terra_signal,
  InternetService,
  Contract,
  tenure,
  feedback_limpo
FROM default.feedback_churn_preco_detalhe
WHERE tipo_motivo_preco = 'Motivo de preço não especificado'
ORDER BY pct_renda_terra_signal DESC;

#### O churn por preço está concentrado em clientes de fibra e contrato mensal, e o principal problema é preço alto, concorrência ou ambos?

In [0]:
%sql
WITH base AS (
  SELECT
    *
  FROM default.feedback_churn_preco_detalhe
  WHERE tipo_motivo_preco <> 'Motivo de preço não especificado'
)

SELECT
  InternetService,
  Contract,

  COUNT(*) AS clientes_churn_preco,

  SUM(preco_percebido_alto) AS clientes_preco_alto,

  SUM(oferta_concorrente) AS clientes_oferta_concorrente,

  ROUND(AVG(monthly_charges), 2) AS mensalidade_media,

  ROUND(
    percentile_approx(monthly_charges, 0.5),
    2
  ) AS mediana_mensalidade,

  ROUND(
    AVG(pct_renda_terra_signal),
    2
  ) AS pct_renda_medio,

  ROUND(
    percentile_approx(pct_renda_terra_signal, 0.5),
    2
  ) AS mediana_pct_renda,

  ROUND(AVG(try_cast(tenure AS DOUBLE)), 1) AS tenure_medio

FROM base
GROUP BY
  InternetService,
  Contract

HAVING COUNT(*) >= 10
ORDER BY clientes_churn_preco DESC;

O churn por preço concentra-se em clientes de fibra óptica com contrato mensal. A principal evidência não é apenas preço alto, mas a facilidade de encontrar uma oferta concorrente percebida como melhor.

- Fibra + month-to-month: 486 casos — concentra a maior parte dos churns por preço.
  - 88,1% citam oferta concorrente (428/486).
  - 36,4% dizem explicitamente que o preço é alto (177/486).
  - A mensalidade média é alta: $89,81.
  - A mediana de comprometimento de renda é só 1,65%, apesar da média de 4,13%. Isso sugere alguns outliers, mas, para a maioria, o problema é competição e valor percebido, não necessariamente falta de renda.
- Fibra + contratos de um/dois anos: menor volume, mas clientes mais maduros e caros.
  - One year: mensalidade média de $101,34 e tenure médio de 51,9 meses.
  - Two year: mensalidade média de $104,66 e tenure médio de 63,9 meses.
  - Aqui a ação é retenção de base valiosa: revisão de preço, benefício de fidelidade e oferta pré-renovação.
- DSL + month-to-month: 48 casos.
  - Apenas 7 citam preço alto.
  - 44 citam concorrente.
  - Mensalidade média bem menor: $50,45.
  - Isso parece mais uma questão de oferta concorrente/percepção de desempenho e valor do que preço absoluto.

In [0]:
%sql
WITH churn_preco_explicito AS (
  SELECT DISTINCT customerID
  FROM default.feedback_churn_preco_detalhe
  WHERE tipo_motivo_preco <> 'Motivo de preço não especificado'
),

base AS (
  SELECT
    h.customerID,
    h.InternetService,
    h.Contract,
    h.Churn,
    CASE
      WHEN p.customerID IS NOT NULL THEN 1
      ELSE 0
    END AS churn_preco_explicito
  FROM default.history_tratado h
  LEFT JOIN churn_preco_explicito p
    ON h.customerID = p.customerID
)

SELECT
  InternetService,
  Contract,

  COUNT(*) AS total_clientes,

  SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS total_churns,

  SUM(churn_preco_explicito) AS churns_por_preco,

  ROUND(
    100.0 * SUM(churn_preco_explicito) / COUNT(*),
    2
  ) AS taxa_churn_preco_no_segmento_pct,

  ROUND(
    100.0 * SUM(churn_preco_explicito)
      / SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END),
    2
  ) AS pct_churns_do_segmento_por_preco

FROM base
GROUP BY
  InternetService,
  Contract
ORDER BY taxa_churn_preco_no_segmento_pct DESC;

A principal descoberta:
Entre clientes de fibra com contrato mensal, 928 de 1.698 cancelaram (54,7% de churn total). Em 486 casos — 28,6% de toda a base desse segmento — o feedback cita explicitamente preço ou oferta concorrente.

Além disso:
- Para fibra + month-to-month, preço/concorrência explica 52,37% dos churns do segmento.
- Para fibra + one year e two year, preço explica cerca de 68% dos churns, embora o volume total seja menor.
- Em DSL mensal, apenas 15% dos churns têm preço como motivo; não seria o primeiro foco para desconto.
- Clientes sem internet cancelam sobretudo por produto/cobertura, não por preço.
Ações recomendadas:
1. Fibra + month-to-month
   Oferta de retenção antes do cancelamento: desconto temporário, preço promocional competitivo ou migração para plano anual com benefício claro.
2. Fibra + one/two year, especialmente clientes antigos
   Campanha de fidelidade e revisão proativa de preço antes da renovação. São menos propensos a sair, mas quando saem, preço/concorrência é frequentemente o motivo.
3. DSL + month-to-month
   Evitar desconto genérico. Testar upgrade de velocidade, pacote de valor ou migração para fibra, porque a concorrência parece pesar mais que o preço absoluto.
4. Clientes sem internet
   Direcionar para expansão de cobertura/oferta, não para ação de preço.

### Qualidade técnica do serviço

In [0]:
%sql
WITH base AS (
  SELECT
    h.customerID,
    h.Churn,
    h.InternetService,
    try_cast(h.tenure AS INT) AS tenure,

    COALESCE(m.motivo_tecnico, 0) AS motivo_tecnico,

    CASE
      WHEN try_cast(h.tenure AS INT) <= 6 THEN '0–6 meses'
      WHEN try_cast(h.tenure AS INT) <= 12 THEN '7–12 meses'
      WHEN try_cast(h.tenure AS INT) <= 36 THEN '13–36 meses'
      ELSE '37+ meses'
    END AS faixa_tenure,

    CASE
      WHEN try_cast(h.tenure AS INT) <= 6 THEN 1
      WHEN try_cast(h.tenure AS INT) <= 12 THEN 2
      WHEN try_cast(h.tenure AS INT) <= 36 THEN 3
      ELSE 4
    END AS ordem_faixa

  FROM default.history_tratado h
  LEFT JOIN default.feedback_churn_motivos m
    ON h.customerID = m.customerID
)

SELECT
  InternetService,
  faixa_tenure,

  COUNT(*) AS total_clientes,

  SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS total_churns,

  SUM(motivo_tecnico) AS churns_com_motivo_tecnico,

  ROUND(
    100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
      / COUNT(*),
    2
  ) AS taxa_churn_total_pct,

  ROUND(
    100.0 * SUM(motivo_tecnico)
      / COUNT(*),
    2
  ) AS taxa_churn_tecnico_no_segmento_pct,

  ROUND(
    100.0 * SUM(motivo_tecnico)
      / NULLIF(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END), 0),
    2
  ) AS pct_dos_churns_por_motivo_tecnico

FROM base
GROUP BY
  InternetService,
  faixa_tenure,
  ordem_faixa

ORDER BY
  InternetService,
  ordem_faixa;

Investigando churn de novos clientes pelo corte de qualidade em internet.

Para DSL, temos:
- DSL, 0–6 meses: 408 clientes; 199 churns; taxa de churn de 48,77%.
- Desses 199 churns, 160 mencionam qualidade técnica: 80,4%.
- Isso significa que 39,22% de todos os novos clientes DSL saíram relatando problema técnico.

O principal problema de retenção nos primeiros seis meses é a qualidade técnica do DSL: lentidão, instabilidade e quedas explicam 80% dos churns desse segmento.

Para fibra, a leitura é diferente:
- Fibra, 0–6 meses: churn total muito alto, 73,51%.
- Porém, apenas 32,12% dos churns citam motivo técnico.
Logo, em fibra recente, qualidade técnica importa, mas não explica a maior parte das saídas. Devemos investigar preço/concorrência, pagamento e contrato nessa fatia.

In [0]:
%sql
WITH churns_recentes AS (
  SELECT
    h.customerID,
    h.InternetService
  FROM default.history_tratado h
  WHERE h.Churn = 'Yes'
    AND try_cast(h.tenure AS INT) <= 6
),

totais AS (
  SELECT
    InternetService,
    COUNT(*) AS total_churns_recentes
  FROM churns_recentes
  GROUP BY InternetService
),

motivos AS (
  SELECT
    c.InternetService,
    m.motivo,
    COUNT(DISTINCT c.customerID) AS clientes_churn
  FROM churns_recentes c
  INNER JOIN default.feedback_churn_motivos_long m
    ON c.customerID = m.customerID
  GROUP BY
    c.InternetService,
    m.motivo
)

SELECT
  m.InternetService,
  m.motivo,
  m.clientes_churn,
  t.total_churns_recentes,

  ROUND(
    100.0 * m.clientes_churn / t.total_churns_recentes,
    2
  ) AS pct_dos_churns_recentes

FROM motivos m
INNER JOIN totais t
  ON m.InternetService = t.InternetService
ORDER BY
  m.InternetService,
  pct_dos_churns_recentes DESC;

QUando observado um recorte de até seis meses, temos contornos bem definidos de estratégia de retenção.

DSL: problema técnico é claramente dominante

- 80,4% dos churns recentes de DSL citam qualidade técnica.
- 48,2% também mencionam pagamento, mas esses motivos se sobrepõem: muitos textos relatam simultaneamente internet lenta/instável e cheque eletrônico inconveniente.
- Preço/concorrência aparece em apenas 3,5%.
Conclusão:
Para novos clientes DSL, não adianta priorizar desconto. O principal problema é experiência técnica: lentidão, instabilidade e quedas logo após a contratação.

Ações recomendadas:
- checagem de qualidade nos primeiros 7, 30 e 60 dias;
- contato proativo se houver falha/baixa performance;
- diagnóstico técnico, troca de equipamento ou upgrade de plano;
- oferta de migração para fibra quando houver cobertura.<br><br>


Fibra: churn inicial alto é multifatorial

Nos churns recentes de fibra:
- 32,1% mencionam qualidade técnica;
- 29,1% mencionam pagamento;
- 26,8% mencionam preço/concorrência;
- 11,5% citam motivo pessoal/financeiro.
Conclusão:
Em fibra, o problema não é único. A retenção inicial deve combinar qualidade de instalação/serviço, oferta competitiva e opções de pagamento.

Ação recomendada: uma jornada de onboarding com diagnóstico técnico, oferta promocional/benefício de fidelidade e convite para pagamento automático ou meios mais convenientes.


Sem internet: falha de produto/cobertura
- 86,7% dos churns recentes de clientes sem internet citam limitação de produto ou cobertura.
Conclusão:
É essencial validar cobertura e necessidade de internet antes da contratação. Esses clientes não devem receber campanha de desconto; precisam de disponibilidade de serviço ou oferta alternativa adequada.

Lembrando: os motivos são multirrótulo, então não se deve somar os percentuais. Eles mostram temas mencionados no mesmo feedback, não causas exclusivas.

Identificamos nos feedbacks relatos de falhas de serviço e também de atendimento. Seria interessante confirmar a causa operacional, e se está sendo agravada por não resposta adequada do atendimento. Para uma avaliação mais completa, seria interessante integrar dados de rede, disponibilidade, incidentes e tickets de suporte aos dados.

### Experiência de pagamento

em quais meios de pagamento o atrito aparece e com quais outros motivos ele se combina

In [0]:
%sql
WITH base AS (
  SELECT
    h.customerID,
    h.PaymentMethod,
    h.Contract,

    CASE
      WHEN h.PaymentMethod IN (
        'Bank transfer (automatic)',
        'Credit card (automatic)'
      ) THEN 'Automático'
      ELSE 'Manual'
    END AS tipo_pagamento,

    m.motivo_preco,
    m.motivo_tecnico,
    m.motivo_produto,
    m.motivo_pessoal

  FROM default.history_tratado h
  INNER JOIN default.feedback_churn_motivos m
    ON h.customerID = m.customerID
  WHERE m.motivo_pagamento = 1
)

SELECT
  PaymentMethod,
  tipo_pagamento,

  COUNT(*) AS churns_com_atrito_pagamento,

  ROUND(
    100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
    2
  ) AS pct_dos_churns_com_atrito_pagamento,

  SUM(motivo_preco) AS tambem_preco_ou_concorrencia,
  SUM(motivo_tecnico) AS tambem_qualidade_tecnica,
  SUM(motivo_produto) AS tambem_produto_ou_cobertura,
  SUM(motivo_pessoal) AS tambem_motivo_pessoal

FROM base
GROUP BY
  PaymentMethod,
  tipo_pagamento
ORDER BY churns_com_atrito_pagamento DESC;

Avaliando o impacto dos meios de pagamento nos casos positivos de churn, temos:
- 94,8% dos churns com atrito de pagamento usam meios manuais (384 de 405).
- Electronic check concentra 54,1% dos casos.
- Mailed check concentra 40,7%.
- Meios automáticos representam apenas 5,2% dos casos.

O atrito de pagamento está concentrado em processos manuais. A empresa deve incentivar migração para débito automático ou cartão recorrente e ampliar opções digitais simples.

Também há sobreposição relevante:
- Em Electronic check, 81 casos também relatam qualidade técnica e 72 preço/concorrência.
- Em Mailed check, 52 também relatam qualidade técnica e 32 limitação de produto/cobertura.
Ou seja, pagamento frequentemente agrava uma experiência já ruim; não deve ser tratado como causa isolada em todos os casos.

Ação prática:
1. Oferecer migração simples de cheque eletrônico/enviado pelo correio para cobrança automática.
2. Criar incentivo de adesão: desconto pontual, crédito na fatura ou benefício de fidelidade.
3. Priorizar clientes de pagamento manual que também apresentem risco técnico ou de preço.
4. Para clientes sem internet/cobertura, não focar em pagamento: o problema principal é oferta de produto.

## Insights da observação dos feedbacks dos clientes


A retenção dos clientes passa por estratédias diversificadas que atendam os clientes de forma customizada.
- DSL recente precisa de correção técnica; 
- fibra mensal precisa de oferta competitiva; 
- pagamento manual precisa de simplificação; 
- e clientes sem internet precisam de produto ou cobertura adequada antes mesmo de qualquer ação promocional.<br>


| Prioridade | Problema identificado | Evidência | Ação prática | Área responsável |
|---|---|---|---|---|
| 1 | Churn precoce em DSL por falha técnica | 80,4% dos churns DSL até 6 meses citam qualidade técnica | Criar jornada de qualidade nos dias 7, 30 e 60; testar conexão, trocar equipamento, acionar suporte proativo e oferecer migração para fibra quando houver cobertura | Operações de rede + Suporte |
| 2 | Fibra mensal perde clientes para concorrência | Em fibra + month-to-month, 28,6% de toda a base do segmento saiu citando preço/oferta concorrente | Oferecer preço promocional, benefício de fidelidade ou migração para contrato anual com vantagem real antes do cancelamento | Comercial + CRM |
| 3 | Clientes antigos de fibra saem por valor percebido | Em fibra com contrato anual/bienal, preço explica cerca de 68% dos churns | Criar campanha pré-renovação: revisão de preço, bônus de velocidade, upgrade ou desconto de permanência | Comercial + Produto |
| 4 | Fibra recente tem churn multifatorial | Até 6 meses: 32,1% técnico, 29,1% pagamento e 26,8% preço/concorrência | Jornada de onboarding: validação pós-instalação, oferta competitiva inicial e estímulo ao pagamento automático | Produto + Operações + CRM |
| 5 | Pagamento manual gera atrito | 94,8% dos churns com atrito de pagamento usam `Electronic check` ou `Mailed check` | Simplificar migração para cartão recorrente/débito automático; oferecer benefício pontual de adesão; ampliar canais digitais | Financeiro + Produto |
| 6 | Produto/cobertura insuficiente | 5,1% dos churns citam ausência de internet, cobertura ou oferta inadequada; 98,7% deles têm `InternetService = No` | Validar cobertura e necessidade de internet antes da venda; evitar vender plano sem conectividade a quem precisa dela; oferecer alternativa compatível | Vendas + Produto + Planejamento de rede |
| 7 | Reclamações técnicas agravadas por atendimento | Parte dos textos associa falha técnica a suporte que não resolveu o problema | Priorizar tickets de instabilidade, criar SLA de resolução e oferecer contato proativo após incidentes | Suporte + Operações |

### AValiando churn por tenure

o churn cai de forma contínua conforme aumenta o tenure?<br>
nos meses 0–3 e 4–6, quais motivos dominam?<br>
depois de um ano, o churn que sobra é mais preço/concorrência, técnico ou produto?<br>

In [0]:
%sql
WITH base AS (
  SELECT
    h.customerID,
    h.Churn,
    try_cast(h.tenure AS INT) AS tenure,

    COALESCE(m.motivo_preco, 0) AS motivo_preco,
    COALESCE(m.motivo_tecnico, 0) AS motivo_tecnico,
    COALESCE(m.motivo_pagamento, 0) AS motivo_pagamento,
    COALESCE(m.motivo_produto, 0) AS motivo_produto,
    COALESCE(m.motivo_pessoal, 0) AS motivo_pessoal,
    COALESCE(m.motivo_atendimento, 0) AS motivo_atendimento,

    CASE
      WHEN try_cast(h.tenure AS INT) <= 3 THEN '0–3 meses'
      WHEN try_cast(h.tenure AS INT) <= 6 THEN '4–6 meses'
      WHEN try_cast(h.tenure AS INT) <= 12 THEN '7–12 meses'
      WHEN try_cast(h.tenure AS INT) <= 24 THEN '13–24 meses'
      WHEN try_cast(h.tenure AS INT) <= 36 THEN '25–36 meses'
      ELSE '37+ meses'
    END AS faixa_tenure,

    CASE
      WHEN try_cast(h.tenure AS INT) <= 3 THEN 1
      WHEN try_cast(h.tenure AS INT) <= 6 THEN 2
      WHEN try_cast(h.tenure AS INT) <= 12 THEN 3
      WHEN try_cast(h.tenure AS INT) <= 24 THEN 4
      WHEN try_cast(h.tenure AS INT) <= 36 THEN 5
      ELSE 6
    END AS ordem_faixa

  FROM default.history_tratado h
  LEFT JOIN default.feedback_churn_motivos m
    ON h.customerID = m.customerID
)

SELECT
  faixa_tenure,

  COUNT(*) AS total_clientes,

  SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS total_churns,

  ROUND(
    100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
      / COUNT(*),
    2
  ) AS taxa_churn_pct,

  -- Motivos citados nos feedbacks de churn
  SUM(motivo_preco) AS churn_preco_concorrencia,
  SUM(motivo_tecnico) AS churn_qualidade_tecnica,
  SUM(motivo_pagamento) AS churn_atrito_pagamento,
  SUM(motivo_produto) AS churn_produto_cobertura,
  SUM(motivo_pessoal) AS churn_pessoal_financeiro,
  SUM(motivo_atendimento) AS churn_atendimento,

  -- Churns cujo feedback não recebeu nenhuma categoria atual
  SUM(
    CASE
      WHEN Churn = 'Yes'
        AND (
          motivo_preco +
          motivo_tecnico +
          motivo_pagamento +
          motivo_produto +
          motivo_pessoal +
          motivo_atendimento
        ) = 0
      THEN 1
      ELSE 0
    END
  ) AS churn_sem_motivo_classificado

FROM base
GROUP BY
  faixa_tenure,
  ordem_faixa
ORDER BY ordem_faixa;

Os principais achados aqui são:
- O churn é máximo no começo: 56,2% até três meses e 43,0% entre quatro e seis meses.
- Até três meses, quase metade dos churns cita qualidade técnica (225/469 = 48,0%) e 39,9% citam atrito de pagamento.
- Entre quatro e seis meses, os problemas técnicos continuam relevantes, mas preço/concorrência passa a ter o mesmo peso.
- A partir de sete meses, preço/concorrência vira o principal tema e cresce proporcionalmente entre os churns.
- Após 37 meses, o churn total é baixo, mas cerca de 63% dos churns ainda citam preço/concorrência (186/294): são clientes antigos que exigem oferta de fidelidade.


Os primeiros seis meses são a janela crítica de retenção. No início, a empresa precisa garantir instalação, desempenho e jornada de pagamento. Após esse período, o desafio se torna reter o cliente por valor percebido, preço competitivo e benefícios de fidelidade.

In [0]:
%sql
-- teste dos sem motivo no grupo de 0-3 meses
SELECT
  h.customerID,
  h.InternetService,
  h.Contract,
  h.PaymentMethod,
  try_cast(h.tenure AS INT) AS tenure,
  try_cast(h.MonthlyCharges AS DECIMAL(12,2)) AS monthly_charges,
  h.CustomerFeedback AS feedback_original

FROM default.history_tratado h
INNER JOIN default.feedback_churn_motivos m
  ON h.customerID = m.customerID

WHERE h.Churn = 'Yes'
  AND try_cast(h.tenure AS INT) <= 3

  AND (
    COALESCE(m.motivo_preco, 0) +
    COALESCE(m.motivo_tecnico, 0) +
    COALESCE(m.motivo_pagamento, 0) +
    COALESCE(m.motivo_produto, 0) +
    COALESCE(m.motivo_pessoal, 0) +
    COALESCE(m.motivo_atendimento, 0)
  ) = 0

ORDER BY rand(42)
LIMIT 20;

In [0]:
%sql
WITH clientes_ate_6_meses AS (
  SELECT
    customerID,
    Churn
  FROM default.history_tratado
  WHERE try_cast(tenure AS INT) <= 6
),

totais AS (
  SELECT
    COUNT(*) AS total_clientes_ate_6_meses,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
      AS total_churns_ate_6_meses
  FROM clientes_ate_6_meses
),

motivos AS (
  SELECT
    m.motivo,
    COUNT(DISTINCT c.customerID) AS clientes_churn
  FROM clientes_ate_6_meses c
  INNER JOIN default.feedback_churn_motivos_long m
    ON c.customerID = m.customerID
  WHERE c.Churn = 'Yes'
  GROUP BY m.motivo
)

SELECT
  m.motivo,
  m.clientes_churn,
  t.total_clientes_ate_6_meses,
  t.total_churns_ate_6_meses,

  ROUND(
    100.0 * m.clientes_churn / t.total_churns_ate_6_meses,
    2
  ) AS pct_dos_churns_ate_6_meses

FROM motivos m
CROSS JOIN totais t
ORDER BY clientes_churn DESC;

Considerando o segmento de tenure de 0-6 meses,
Cerca de um quinto da base gera mais de dois quintos dos cancelamentos. É o segmento prioritário para retenção.

- 1.179 clientes têm até 6 meses de tenure, que é 20,9% da base total (1.179 / 5.634).
- 617 com churn positivo, que representam 41% de todos os churns (617 / 1.504).
- A taxa de churn desse grupo é 52,3%.


A ação prática para esse grupo prevê uma jornada de retenção nesses primeiros seis meses:
1. Garantir qualidade e estabilidade da conexão, principalmente para DSL.
2. Simplificar pagamento e incentivar meios automáticos.
3. Fazer oferta competitiva para fibra mensal. (aqui em especial a partir do terceiro/quarto mês)
4. Validar cobertura e necessidade de internet antes da venda.